# S6E4 — 04 XGBoost seed ensemble + class weights (intermediate submission)

> Teammate's notebook from the team's Kaggle pipeline, cleaned for the repository: empty cells, `!ls` calls, duplicates and commented-out code removed; the code that ran is unchanged.

| | |
|---|---|
| **Input** | OOF and test probabilities of XGBoost trained with seeds 0, 1, 2 (training notebooks were run on Kaggle and are not included) |
| **Method** | mean of the 3 seeds → per-class weights on the probabilities (cost-sensitive argmax), tuned on OOF for Balanced Accuracy |
| **Result** | OOF Balanced Accuracy 0.97894 (argmax) → **0.98010** (weights 0.396 / 0.494 / 1.474) |

**Known issue, kept as in the original:** the threshold cell near the end averages the test probabilities over *rows* (`mean(axis=0)`); rerun as is, it would give every row the same class. The class-weight method is the one carried to the final stack (`05`).

In [1]:
import pandas as pd
import numpy as np

In [2]:
oof_xgb_0 = np.load("/kaggle/input/notebooks/prashantlimba/seed-0-s6e4-highest-score-xgboost-cv/oof_preds_xgb.npy")
test_preds_0 = np.load("/kaggle/input/notebooks/prashantlimba/seed-0-s6e4-highest-score-xgboost-cv/test_preds_xgb.npy")

oof_xgb_1 = np.load("/kaggle/input/notebooks/prashantlimba/seed-1-s6e4-highest-score-xgboost-cv/oof_preds_xgb.npy")
test_preds_1 = np.load("/kaggle/input/notebooks/prashantlimba/seed-1-s6e4-highest-score-xgboost-cv/test_preds_xgb.npy")

oof_xgb_2 = np.load("/kaggle/input/notebooks/prashantlimba/seed-2-s6e4-highest-score-xgboost-cv/oof_preds_xgb.npy")
test_preds_2 = np.load("/kaggle/input/notebooks/prashantlimba/seed-2-s6e4-highest-score-xgboost-cv/test_preds_xgb.npy")

In [3]:
oof_xgb = (oof_xgb_0 + oof_xgb_1 + oof_xgb_2) / 3
test_preds = (test_preds_0 + test_preds_1 + test_preds_2) / 3

In [4]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e4/train.csv")

In [5]:
train['Irrigation_Need'] = train['Irrigation_Need'].map({'Low': 0, 'Medium': 1, 'High': 2})
train = train[['id', 'Irrigation_Need']]
train.head()

,id,Irrigation_Need
0,0,0
1,1,0
2,2,0
3,3,1
4,4,0


In [6]:
oof_xgb.shape

(630000, 3)

In [7]:
import numpy as np
from sklearn.metrics import balanced_accuracy_score
from itertools import product

def cutoff_analysis(y_true, y_probs, n_thresholds=50):
    """
    Find optimal decision thresholds for 3-class classification
    to maximize balanced accuracy.
    
    Parameters
    ----------
    y_true : array-like, shape (n_samples,)
        True labels (0, 1, 2).
    y_probs : array-like, shape (n_samples, 3)
        Predicted probabilities for each class.
    n_thresholds : int
        Number of threshold values to search per class pair.
    
    Returns
    -------
    best_thresholds : dict
        Optimal thresholds found.
    best_bal_acc : float
        Best balanced accuracy achieved.
    results : list of dict
        All evaluated combinations sorted descending by balanced accuracy.
    """
    y_true = np.asarray(y_true)
    y_probs = np.asarray(y_probs)
    n_classes = y_probs.shape[1]

    # --- Method 1: Threshold on class probabilities ---
    # For each class, sweep a threshold; assign the class with
    # highest prob that exceeds its threshold, else argmax fallback.
    thresholds = np.linspace(0.05, 0.95, n_thresholds)

    # Coarse grid search over all 3 class thresholds
    # Full grid is too large, so do coordinate descent
    best_t = [0.5] * n_classes
    best_bal_acc = 0.0

    for iteration in range(5):  # coordinate descent rounds
        for cls in range(n_classes):
            best_cls_t = best_t[cls]
            for t in thresholds:
                trial_t = best_t.copy()
                trial_t[cls] = t
                preds = _apply_thresholds(y_probs, trial_t)
                ba = balanced_accuracy_score(y_true, preds)
                if ba > best_bal_acc:
                    best_bal_acc = ba
                    best_cls_t = t
            best_t[cls] = best_cls_t

    # Fine-tune around the best found thresholds
    fine_range = 0.05
    fine_n = 20
    for iteration in range(3):
        for cls in range(n_classes):
            lo = max(0.01, best_t[cls] - fine_range)
            hi = min(0.99, best_t[cls] + fine_range)
            fine_thresholds = np.linspace(lo, hi, fine_n)
            best_cls_t = best_t[cls]
            for t in fine_thresholds:
                trial_t = best_t.copy()
                trial_t[cls] = t
                preds = _apply_thresholds(y_probs, trial_t)
                ba = balanced_accuracy_score(y_true, preds)
                if ba > best_bal_acc:
                    best_bal_acc = ba
                    best_cls_t = t
            best_t[cls] = best_cls_t
        fine_range /= 2

    # --- Method 2: Cost-sensitive argmax (scale probs) ---
    # Multiply each class prob by a weight before argmax
    weights_range = np.linspace(0.2, 5.0, n_thresholds)
    best_weights = [1.0] * n_classes
    best_bal_acc_w = 0.0

    for iteration in range(5):
        for cls in range(n_classes):
            best_cls_w = best_weights[cls]
            for w in weights_range:
                trial_w = best_weights.copy()
                trial_w[cls] = w
                scaled = y_probs * np.array(trial_w)
                preds = scaled.argmax(axis=1)
                ba = balanced_accuracy_score(y_true, preds)
                if ba > best_bal_acc_w:
                    best_bal_acc_w = ba
                    best_cls_w = w
            best_weights[cls] = best_cls_w

    # Pick the better method
    if best_bal_acc_w > best_bal_acc:
        print(f"Best method: Cost-sensitive argmax")
        print(f"  Class weights: {[f'{w:.4f}' for w in best_weights]}")
        print(f"  Balanced Accuracy: {best_bal_acc_w:.6f}")
        scaled = y_probs * np.array(best_weights)
        best_preds = scaled.argmax(axis=1)
        return {
            "method": "cost_sensitive_argmax",
            "weights": best_weights,
            "balanced_accuracy": best_bal_acc_w,
            "predictions": best_preds,
        }
    else:
        print(f"Best method: Probability thresholds")
        print(f"  Thresholds: {[f'{t:.4f}' for t in best_t]}")
        print(f"  Balanced Accuracy: {best_bal_acc:.6f}")
        best_preds = _apply_thresholds(y_probs, best_t)
        return {
            "method": "probability_thresholds",
            "thresholds": best_t,
            "balanced_accuracy": best_bal_acc,
            "predictions": best_preds,
        }


def _apply_thresholds(y_probs, thresholds):
    """
    Assign class based on thresholds: for each sample, consider only
    classes whose probability exceeds the threshold, then pick the
    one with the highest probability. Fallback to argmax if none exceed.
    """
    t = np.array(thresholds)
    exceeds = y_probs >= t  # (n_samples, n_classes)
    # Mask probs that don't meet threshold
    masked = np.where(exceeds, y_probs, -1.0)
    # If no class exceeds its threshold, fall back to raw argmax
    no_exceed = ~exceeds.any(axis=1)
    masked[no_exceed] = y_probs[no_exceed]
    return masked.argmax(axis=1)


# --- Compare against default argmax baseline ---
def run_analysis(y_true, y_probs):
    y_true = np.asarray(y_true)
    y_probs = np.asarray(y_probs)

    baseline_preds = y_probs.argmax(axis=1)
    baseline_ba = balanced_accuracy_score(y_true, baseline_preds)
    print(f"Baseline (argmax) Balanced Accuracy: {baseline_ba:.6f}")
    print(f"Class distribution: {dict(zip(*np.unique(y_true, return_counts=True)))}")
    print()

    result = cutoff_analysis(y_true, y_probs)

    improvement = result["balanced_accuracy"] - baseline_ba
    print(f"\nImprovement over baseline: {improvement:+.6f}")

    return result


# === Usage ===
# result = run_analysis(y_true, y_probs)
# optimized_predictions = result["predictions"]

In [8]:
result = run_analysis(train['Irrigation_Need'], oof_xgb)

Baseline (argmax) Balanced Accuracy: 0.978936
Class distribution: {np.int64(0): np.int64(369917), np.int64(1): np.int64(239074), np.int64(2): np.int64(21009)}

Best method: Cost-sensitive argmax
  Class weights: ['0.3959', '0.4939', '1.4735']
  Balanced Accuracy: 0.980098

Improvement over baseline: +0.001161


In [ ]:
test_preds.shape

In [9]:
pred_xgb = test_preds
weights = np.array([0.3959, 0.4939, 1.4735])
scaled = pred_xgb * weights
optimized_predictions = scaled.argmax(axis=1)

In [ ]:
import numpy as np

# ---- Paste your learned thresholds here ----
BEST_THRESHOLDS = [0.4902, 0.9225, 0.0795]


def apply_thresholds_inference(y_probs, thresholds=BEST_THRESHOLDS):
    """
    Apply optimized probability thresholds to model outputs.
    
    Parameters
    ----------
    y_probs : array-like, shape (n_samples, n_classes)
        Predicted probabilities from model.
    thresholds : list
        Learned thresholds per class.
    
    Returns
    -------
    preds : np.ndarray
        Final predicted classes.
    """
    y_probs = np.asarray(y_probs)
    t = np.array(thresholds)

    exceeds = y_probs >= t
    masked = np.where(exceeds, y_probs, -1.0)

    # fallback to argmax if no class crosses threshold
    no_exceed = ~exceeds.any(axis=1)
    masked[no_exceed] = y_probs[no_exceed]

    return masked.argmax(axis=1)

In [ ]:
# Step 2: apply thresholds
optimized_predictions = apply_thresholds_inference(pred_xgb.mean(axis=0))

In [10]:
TARGET='Irrigation_Need'
submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv')
submission[TARGET] = optimized_predictions
submission[TARGET] = submission[TARGET].map({0: 'Low', 1: 'Medium', 2: 'High'})
submission.to_csv(f'cutoff_test_pred_xgb_v8.csv', index=False)

In [11]:
submission[TARGET].value_counts()

Irrigation_Need
Low       159281
Medium    100447
High       10272
Name: count, dtype: int64